 # 06 Agentic Energy Controller

Runs the telemetry-driven client controller against the matched stage-05 all-client baseline. The experiment uses real preprocessed client splits, model-derived loss/F1 telemetry, persisted energy state, and round-level resume.

In [1]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
EXPECTED_DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = EXPECTED_DRIVE_PROJECT_ROOT


def is_project_root(path):
    return (path / 'configs' / 'project_colab.json').is_file() and (path / 'src' / 'ecotrust').is_dir()


def discover_mounted_project():
    candidates = [EXPECTED_DRIVE_PROJECT_ROOT]
    my_drive = Path('/content/drive/MyDrive')
    candidates.extend([
        my_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
        my_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
    ])
    shared_drives = Path('/content/drive/Shareddrives')
    if shared_drives.is_dir():
        for shared_drive in shared_drives.iterdir():
            candidates.extend([
                shared_drive / 'projects' / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
            ])
    return next((candidate for candidate in candidates if is_project_root(candidate)), None)


try:
    from google.colab import drive  # type: ignore
    if not is_project_root(EXPECTED_DRIVE_PROJECT_ROOT):
        print('Drive project is not visible; resetting stale DriveFS state...', flush=True)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        subprocess.run(
            ['fusermount', '-uz', '/content/drive'],
            check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        shutil.rmtree('/content/drive', ignore_errors=True)
        Path('/content/drive').mkdir(parents=True, exist_ok=True)
        drive.mount('/content/drive', force_remount=False, timeout_ms=120000)
    discovered_root = discover_mounted_project()
    if discovered_root is not None:
        DRIVE_PROJECT_ROOT = discovered_root
except ImportError:
    pass
except Exception as exc:
    raise RuntimeError(f'Google Drive project mount failed: {type(exc).__name__}: {exc}') from exc


def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError(
        'Cannot find EcoTrust-FLIDS IMPLEMENTATION in the mounted Google Drive. Expected path: '
        + str(EXPECTED_DRIVE_PROJECT_ROOT) + '. Authorize the Google account that can open '
        'https://drive.google.com/drive/folders/1lWrVkEURlFciIae0chK8hjDUpVafGWkS, '
        'or add that folder as a shortcut under MyDrive/projects/implementation.'
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if str(PROJECT_ROOT).startswith('/content/drive') and (PROJECT_ROOT / 'configs' / 'project_colab.json').exists():
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project_colab.json'
else:
    CONFIG_PATH = PROJECT_ROOT / 'configs' / 'project.json'

RUN_MODE = 'full'
DEVICE = 'cpu'
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
BUDGET_FRACTION = 0.6

importlib.invalidate_caches()
import ecotrust.agent.experiment as agent_experiment_module
importlib.reload(agent_experiment_module)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)
print('RUN_MODE:', RUN_MODE)
print('DEVICE:', DEVICE)
print('AGENT_EXPERIMENT_SOURCE:', agent_experiment_module.__file__)

result = agent_experiment_module.run_agentic_energy_experiment(
    CONFIG_PATH,
    run_mode=RUN_MODE,
    datasets=DATASETS,
    device=DEVICE,
    budget_fraction=BUDGET_FRACTION,
    resume=True,
)
print('STAGE_06_STATUS:', result['status'])
print('STAGE_06_MANIFEST:', result['manifest'])


PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
RUN_MODE: full
DEVICE: cpu
AGENT_EXPERIMENT_SOURCE: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/src/ecotrust/agent/experiment.py
Agentic federated Edge-IIoTset
[2026-10-23T07:50:27+00:00] Starting 30 federated rounds on cpu; model_size_mb=0.1426.
[2026-10-23T07:50:33+00:00] Round 1/30 complete: f1=1.000000, selected=12/20, energy_saving=33.00%.
[2026-10-23T07:50:37+00:00] Round 2/30 complete: f1=1.000000, selected=12/20, energy_saving=33.00%.
[2026-10-23T07:50:41+00:00] Round 3/30 complete: f1=1.000000, selected=12/20, energy_saving=33.00%.
[2026-10-23T07:50:46+00:00] Round 4/30 complete: f1=1.000000, selected=12/20, energy_saving=33.00%.
[2026-10-23T07:50:49+00:00] Round 5/30 complete: f1=1.000000, selected=12/20, e